In [1]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def load_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    # Enforce numeric types
    for col in ["seed", "n", "m", "y"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    for col in ["phi", "aHat", "absErr", "aTrue", "cpu_ms"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop rows with critical missing values
    df = df.dropna(subset=["seed", "n", "m", "aHat", "absErr", "aTrue", "cpu_ms"]).copy()

    # Extra derived metrics
    df["relErr"] = df["absErr"] / df["aTrue"].replace(0, np.nan)
    df["log_cpu_ms"] = np.log10(df["cpu_ms"].clip(lower=1e-9))
    return df


def summarize(df: pd.DataFrame) -> None:
    print("\n=== Basic info ===")
    print(f"Rows: {len(df)}")
    print(f"Seeds: {sorted(df['seed'].dropna().unique().tolist())}")
    print(f"n values: {sorted(df['n'].dropna().unique().tolist())}")
    print(f"m values: {sorted(df['m'].dropna().unique().tolist())}")

    print("\n=== Overall summary ===")
    overall = df.agg(
        mean_absErr=("absErr", "mean"),
        median_absErr=("absErr", "median"),
        mean_relErr=("relErr", "mean"),
        median_relErr=("relErr", "median"),
        mean_cpu_ms=("cpu_ms", "mean"),
        median_cpu_ms=("cpu_ms", "median"),
    )
    print(overall.to_string())

    print("\n=== Summary by (n, m) ===")
    grp = (
        df.groupby(["n", "m"], as_index=False)
        .agg(
            count=("absErr", "size"),
            mean_absErr=("absErr", "mean"),
            median_absErr=("absErr", "median"),
            std_absErr=("absErr", "std"),
            mean_relErr=("relErr", "mean"),
            mean_cpu_ms=("cpu_ms", "mean"),
            median_cpu_ms=("cpu_ms", "median"),
            std_cpu_ms=("cpu_ms", "std"),
        )
        .sort_values(["n", "m"])
    )
    print(grp.to_string(index=False))

    # Also show best m per n (lowest mean absErr)
    best = grp.loc[grp.groupby("n")["mean_absErr"].idxmin()].sort_values("n")
    print("\n=== Best m (lowest mean absErr) per n ===")
    print(best[["n", "m", "mean_absErr", "mean_cpu_ms"]].to_string(index=False))

In [7]:

def detect_timing_inversions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flag cases where cpu_ms decreases when m increases (per seed,n).
    This often happens due to warm-up, GC, or measurement artifacts.
    """
    inv_rows = []
    for (seed, n), g in df.groupby(["seed", "n"]):
        g2 = g.sort_values("m").reset_index(drop=True)
        ms = g2["cpu_ms"].to_numpy()
        mvals = g2["m"].astype(int).to_numpy()

        for i in range(1, len(g2)):
            if mvals[i] > mvals[i - 1] and ms[i] < ms[i - 1]:
                inv_rows.append(
                    {
                        "seed": int(seed),
                        "n": int(n),
                        "m_prev": int(mvals[i - 1]),
                        "cpu_prev": float(ms[i - 1]),
                        "m_curr": int(mvals[i]),
                        "cpu_curr": float(ms[i]),
                        "drop_ms": float(ms[i - 1] - ms[i]),
                        "drop_pct": float((ms[i - 1] - ms[i]) / max(ms[i - 1], 1e-12) * 100.0),
                    }
                )

    inv = pd.DataFrame(inv_rows)
    if len(inv) == 0:
        print("\n=== Timing inversions ===")
        print("None detected.")
        return inv

    inv = inv.sort_values(["drop_ms"], ascending=False)
    print("\n=== Timing inversions (cpu_ms decreased when m increased) ===")
    print(inv.to_string(index=False))
    return inv


def detect_error_trend(df: pd.DataFrame) -> pd.DataFrame:
    """
    Check if absErr generally decreases with m (per seed,n). This should often improve with m.
    """
    bad = []
    for (seed, n), g in df.groupby(["seed", "n"]):
        g2 = g.sort_values("m").reset_index(drop=True)
        mvals = g2["m"].astype(int).to_numpy()
        errs = g2["absErr"].to_numpy()
        # count how many times error increases when m increases
        inc = 0
        for i in range(1, len(g2)):
            if mvals[i] > mvals[i - 1] and errs[i] > errs[i - 1] + 1e-15:
                inc += 1
        bad.append({"seed": int(seed), "n": int(n), "num_increases": int(inc), "points": int(len(g2))})

    bad_df = pd.DataFrame(bad).sort_values(["num_increases", "n", "seed"], ascending=[False, True, True])
    print("\n=== absErr trend check (per seed,n): how often absErr increased as m increased ===")
    print(bad_df.to_string(index=False))
    return bad_df

In [ ]:
def plot_absErr_vs_m(df: pd.DataFrame, outdir: Path) -> None:
    outdir.mkdir(parents=True, exist_ok=True)

    # Plot mean absErr vs m for each n
    grp = (
        df.groupby(["n", "m"], as_index=False)
        .agg(mean_absErr=("absErr", "mean"), mean_cpu_ms=("cpu_ms", "mean"))
        .sort_values(["n", "m"])
    )

    plt.figure()
    for n, g in grp.groupby("n"):
        plt.plot(g["m"], g["mean_absErr"], marker="o", label=f"n={int(n)}")
    plt.yscale("log")
    plt.xlabel("m (control qubits)")
    plt.ylabel("mean absErr (log scale)")
    plt.title("Amplitude Estimation error vs m (mean over seeds)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outdir / "absErr_vs_m.png", dpi=200)
    plt.close()

    # Plot mean cpu_ms vs m for each n
    plt.figure()
    for n, g in grp.groupby("n"):
        plt.plot(g["m"], g["mean_cpu_ms"], marker="o", label=f"n={int(n)}")
    plt.yscale("log")
    plt.xlabel("m (control qubits)")
    plt.ylabel("mean cpu_ms (log scale)")
    plt.title("Runtime vs m (mean over seeds)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outdir / "cpu_ms_vs_m.png", dpi=200)
    plt.close()


def save_tables(df: pd.DataFrame, inv: pd.DataFrame, outdir: Path) -> None:
    outdir.mkdir(parents=True, exist_ok=True)

    # Summary by (seed,n,m)
    df_sorted = df.sort_values(["seed", "n", "m"])
    df_sorted.to_csv(outdir / "cleaned_sorted.csv", index=False)

    # Aggregate table by (n,m)
    agg = (
        df.groupby(["n", "m"], as_index=False)
        .agg(
            count=("absErr", "size"),
            mean_absErr=("absErr", "mean"),
            median_absErr=("absErr", "median"),
            std_absErr=("absErr", "std"),
            mean_relErr=("relErr", "mean"),
            mean_cpu_ms=("cpu_ms", "mean"),
            median_cpu_ms=("cpu_ms", "median"),
            std_cpu_ms=("cpu_ms", "std"),
        )
        .sort_values(["n", "m"])
    )
    agg.to_csv(outdir / "summary_by_n_m.csv", index=False)

    if inv is not None and len(inv) > 0:
        inv.to_csv(outdir / "timing_inversions.csv", index=False)

In [9]:
csv_path = Path("results_finalSV.csv")
outdir = Path("analysis_results")
df = load_csv(csv_path)
summarize(df)
inv = detect_timing_inversions(df)
detect_error_trend(df)
plot_absErr_vs_m(df, outdir)
save_tables(df, inv, outdir)



=== Basic info ===
Rows: 54
Seeds: [2, 4, 42]
n values: [3, 4, 5]
m values: [5, 6, 7, 8, 9, 10]

=== Overall summary ===
                 absErr    relErr       cpu_ms
mean_absErr    0.031181       NaN          NaN
median_absErr  0.003461       NaN          NaN
mean_relErr         NaN  0.784904          NaN
median_relErr       NaN  0.055370          NaN
mean_cpu_ms         NaN       NaN  31125.00313
median_cpu_ms       NaN       NaN   2350.24200

=== Summary by (n, m) ===
 n  m  count  mean_absErr  median_absErr  std_absErr  mean_relErr   mean_cpu_ms  median_cpu_ms   std_cpu_ms
 3  5      3     0.027876       0.021447    0.011136     0.223008     16.073667         16.241     2.111978
 3  6      3     0.014819       0.011505    0.005740     0.118552     59.573667         62.244     5.962259
 3  7      3     0.006851       0.004524    0.004030     0.054811    217.413667        222.788    13.141324
 3  8      3     0.034634       0.004524    0.052949     0.277069    894.789000        881